# TranPy Dataset Regeneration - Google Colab

This notebook regenerates TranPy datasets with the current pandas version to fix compatibility issues.

**What this does:**
1. Downloads old pickle files from Google Drive
2. Loads them with compatibility handling
3. Regenerates with current pandas version
4. Creates HDF5 and Parquet versions (recommended)

**Run this in Google Colab for best results!**

## Setup - Check Versions

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print("=" * 60)
print("TranPy Dataset Regeneration")
print("=" * 60)
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"Python version: {__import__('sys').version}")
print("=" * 60)

TranPy Dataset Regeneration
pandas version: 2.3.3
numpy version: 2.3.3
Python version: 3.13.6 (main, Aug  8 2025, 16:50:55) [Clang 20.1.4 ]


## Step 1: Install Dependencies

In [2]:
# Install gdown for Google Drive downloads
!pip install gdown -q
!pip install tables -q  # For HDF5 support
!pip install pyarrow -q  # For Parquet support

print("✓ Dependencies installed")

zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip
✓ Dependencies installed


## Step 2: Download Datasets from Google Drive

In [5]:
import gdown

# Create directories
Path('original_datasets').mkdir(exist_ok=True)
Path('regenerated_datasets').mkdir(exist_ok=True)

# Dataset metadata
datasets = {
    'NewEngland': {
        'file_id': '1eXtw44VXhYM0jQyJGGY5Eevdrg8yui0w',
        'filename': 'NewEngland.pickle'
    },
    'NineBusSystem': {
        'file_id': '1-4LrEqmDP6-EcLpL0-6tvzsNSSJOLzG1',
        'filename': 'NineBusSystem.pickle'
    }
}

# Download files
print("📥 Downloading datasets from Google Drive...\n")

for name, info in datasets.items():
    dest = Path('original_datasets') / info['filename']
    
    if dest.exists():
        print(f"✓ {name} already downloaded")
        continue
    
    print(f"Downloading {name}...")
    url = f"https://drive.google.com/uc?id={info['file_id']}"
    
    try:
        gdown.download(url, str(dest), quiet=False)
        print(f"✓ Downloaded: {name}\n")
    except Exception as e:
        print(f"❌ Error downloading {name}: {e}\n")

print("\n✅ Download complete!")

ModuleNotFoundError: No module named 'gdown'

## Step 3: Define Loading Functions with Compatibility

In [ ]:
def load_old_pickle_with_compatibility(filepath):
    """
    Load old pickle file with pandas compatibility handling.
    
    This handles the pandas.core.indexes.numeric deprecation issue.
    """
    import sys
    import pandas.core.indexes.numeric as numeric_index
    
    # Create compatibility mapping
    sys.modules['pandas.core.indexes.numeric'] = numeric_index
    
    try:
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        print(f"✓ Loaded: {filepath.name}")
        return data
    except Exception as e:
        print(f"❌ Error loading {filepath}: {e}")
        raise


def inspect_dataset(data, name):
    """Inspect dataset structure."""
    print(f"\n📊 Inspecting {name}:")
    print(f"  Type: {type(data)}")
    
    if hasattr(data, '__dict__'):
        attrs = list(data.__dict__.keys())
        print(f"  Attributes: {attrs}")
    
    # Check for common attributes
    if hasattr(data, 'bus_data_post_fault'):
        print(f"  ✓ Has bus_data_post_fault")
        if isinstance(data.bus_data_post_fault, (list, tuple)):
            print(f"    Length: {len(data.bus_data_post_fault)}")
    
    if hasattr(data, 'df_events'):
        print(f"  ✓ Has df_events")
        if hasattr(data.df_events, 'shape'):
            print(f"    Shape: {data.df_events.shape}")
            print(f"    Columns: {list(data.df_events.columns)}")

print("✓ Functions defined")

## Step 4: Regenerate Pickle Files

In [ ]:
print("🔄 Regenerating pickle files with current pandas...\n")

regenerated_data = {}

for name, info in datasets.items():
    original_path = Path('original_datasets') / info['filename']
    
    if not original_path.exists():
        print(f"⚠️  Skipping {name} - file not found\n")
        continue
    
    print(f"Processing {name}...")
    
    try:
        # Load old pickle
        data = load_old_pickle_with_compatibility(original_path)
        
        # Inspect
        inspect_dataset(data, name)
        
        # Save with current pandas version
        output_path = Path('regenerated_datasets') / info['filename']
        with open(output_path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        print(f"✓ Saved: {output_path}")
        
        # Verify loading
        with open(output_path, 'rb') as f:
            verified = pickle.load(f)
        print(f"✓ Verified loading works")
        
        # Store for later conversion
        regenerated_data[name] = data
        
        print(f"\n✅ {name} pickle regenerated successfully!\n")
        print("=" * 60 + "\n")
        
    except Exception as e:
        print(f"❌ Failed to regenerate {name}: {e}\n")
        print("=" * 60 + "\n")

print("✅ Pickle regeneration complete!")

## Step 5: Convert to HDF5 Format (RECOMMENDED)

In [ ]:
def convert_to_hdf5(data, output_path, grid_name):
    """
    Convert dataset to HDF5 format (better long-term compatibility).
    
    HDF5 advantages:
    - Language-agnostic (R, Julia, MATLAB, etc.)
    - No pickle compatibility issues
    - Efficient compression
    """
    print(f"💾 Converting {grid_name} to HDF5...")
    
    # Extract data
    voltage_angle_data = data.bus_data_post_fault[1]
    events_df = data.df_events
    
    # Parse voltage and angle measurements
    voltages = []
    angles = []
    
    for event_data in voltage_angle_data:
        dict_temp = list(event_data.values())[-1]
        event_voltages = []
        event_angles = []
        
        for key in dict_temp.keys():
            if 'm:u' in key:
                event_voltages.append(dict_temp[key])
            elif 'm:ph' in key:
                event_angles.append(dict_temp[key])
        
        voltages.append(event_voltages)
        angles.append(event_angles)
    
    # Create DataFrames
    voltage_df = pd.DataFrame(voltages)
    angle_df = pd.DataFrame(angles)
    features_df = pd.concat([voltage_df, angle_df], axis=1)
    
    # Add proper column names
    n_buses = len(voltages[0])
    col_names = [f'bus_{i}_voltage' for i in range(1, n_buses + 1)]
    col_names += [f'bus_{i}_angle' for i in range(1, n_buses + 1)]
    features_df.columns = col_names
    
    # Add labels
    features_df['stability'] = events_df['system_stability'].values
    
    # Save to HDF5
    features_df.to_hdf(output_path, key='data', mode='w', complevel=9)
    
    print(f"✓ Saved HDF5: {output_path.name}")
    print(f"  Shape: {features_df.shape}")
    print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Save metadata
    metadata = {
        'grid_name': grid_name,
        'n_samples': len(features_df),
        'n_features': len(features_df.columns) - 1,
        'n_buses': n_buses,
        'pandas_version': pd.__version__,
        'numpy_version': np.__version__
    }
    pd.DataFrame([metadata]).to_hdf(output_path, key='metadata', mode='a')
    
    # Verify
    loaded_df = pd.read_hdf(output_path, key='data')
    print(f"✓ Verified HDF5 loading: {loaded_df.shape}")
    
    return features_df


# Convert all datasets
print("💾 Converting datasets to HDF5 format...\n")

for name, data in regenerated_data.items():
    output_path = Path('regenerated_datasets') / f"{name}.h5"
    
    try:
        convert_to_hdf5(data, output_path, name)
        print(f"✅ {name} HDF5 created!\n")
    except Exception as e:
        print(f"❌ HDF5 conversion failed for {name}: {e}\n")

print("✅ HDF5 conversion complete!")

## Step 6: Convert to Parquet Format (ALTERNATIVE)

In [ ]:
def convert_to_parquet(data, output_path, grid_name):
    """
    Convert dataset to Parquet format (smaller files).
    
    Parquet advantages:
    - Smaller file size
    - Fast columnar access
    - Good compression
    """
    print(f"💾 Converting {grid_name} to Parquet...")
    
    # Extract data (same as HDF5)
    voltage_angle_data = data.bus_data_post_fault[1]
    events_df = data.df_events
    
    voltages = []
    angles = []
    
    for event_data in voltage_angle_data:
        dict_temp = list(event_data.values())[-1]
        event_voltages = [v for k, v in dict_temp.items() if 'm:u' in k]
        event_angles = [v for k, v in dict_temp.items() if 'm:ph' in k]
        voltages.append(event_voltages)
        angles.append(event_angles)
    
    # Create DataFrame
    features_df = pd.concat([
        pd.DataFrame(voltages),
        pd.DataFrame(angles)
    ], axis=1)
    
    # Add column names
    n_buses = len(voltages[0])
    col_names = [f'bus_{i}_voltage' for i in range(1, n_buses + 1)]
    col_names += [f'bus_{i}_angle' for i in range(1, n_buses + 1)]
    features_df.columns = col_names
    
    # Add labels
    features_df['stability'] = events_df['system_stability'].values
    
    # Save to Parquet
    features_df.to_parquet(output_path, compression='snappy', index=False)
    
    print(f"✓ Saved Parquet: {output_path.name}")
    print(f"  Shape: {features_df.shape}")
    print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
    
    # Verify
    loaded_df = pd.read_parquet(output_path)
    print(f"✓ Verified Parquet loading: {loaded_df.shape}")
    
    return features_df


# Convert all datasets
print("💾 Converting datasets to Parquet format...\n")

for name, data in regenerated_data.items():
    output_path = Path('regenerated_datasets') / f"{name}.parquet"
    
    try:
        convert_to_parquet(data, output_path, name)
        print(f"✅ {name} Parquet created!\n")
    except Exception as e:
        print(f"❌ Parquet conversion failed for {name}: {e}\n")

print("✅ Parquet conversion complete!")

## Step 7: Summary and File Sizes

In [ ]:
print("=" * 60)
print("✅ DATASET REGENERATION COMPLETE!")
print("=" * 60)

print("\n📊 File Size Comparison:\n")

for name in datasets.keys():
    print(f"{name}:")
    
    # Original
    orig = Path('original_datasets') / f"{name}.pickle"
    if orig.exists():
        print(f"  Original pickle:     {orig.stat().st_size / 1024 / 1024:6.2f} MB")
    
    # Regenerated pickle
    regen_pkl = Path('regenerated_datasets') / f"{name}.pickle"
    if regen_pkl.exists():
        print(f"  Regenerated pickle:  {regen_pkl.stat().st_size / 1024 / 1024:6.2f} MB")
    
    # HDF5
    hdf5 = Path('regenerated_datasets') / f"{name}.h5"
    if hdf5.exists():
        print(f"  HDF5:                {hdf5.stat().st_size / 1024 / 1024:6.2f} MB")
    
    # Parquet
    parquet = Path('regenerated_datasets') / f"{name}.parquet"
    if parquet.exists():
        print(f"  Parquet:             {parquet.stat().st_size / 1024 / 1024:6.2f} MB")
    
    print()

print("=" * 60)
print("\n📋 RECOMMENDATIONS:")
print("\n1. ✅ Use HDF5 (.h5) for best compatibility")
print("   - No pickle issues")
print("   - Language-agnostic")
print("   - Good compression")
print("\n2. ⚡ Use Parquet for smallest size")
print("   - Smaller files")
print("   - Fast access")
print("   - Modern standard")
print("\n3. 🔧 Use regenerated pickle if needed")
print("   - Compatible with current pandas")
print("   - Drop-in replacement")

print("\n📦 Next steps:")
print("  1. Download files from 'regenerated_datasets/' folder")
print("  2. Replace in: src/tranpy/data/datasets/")
print("  3. Test: from tranpy.datasets import load_newengland")
print("\n" + "=" * 60)

## Step 8: Download Files (Click to download)

In [ ]:
from google.colab import files
import zipfile

print("📦 Creating ZIP archive...")

# Create ZIP file
with zipfile.ZipFile('regenerated_datasets.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in Path('regenerated_datasets').iterdir():
        if file.is_file():
            zipf.write(file, arcname=file.name)
            print(f"  Added: {file.name}")

print("\n✓ ZIP created: regenerated_datasets.zip")
print("\n📥 Downloading...")

files.download('regenerated_datasets.zip')

print("\n✅ Download complete!")
print("\nExtract the ZIP and replace files in:")
print("  src/tranpy/data/datasets/")